In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy")

Spark 4.0.0-preview2 — gotowy


In [2]:
df = spark.read.json("transactions_10k.jsonl")

print(f"Liczba rekordów: {df.count()}")
df.printSchema()
df.show(10, truncate=False)

Liczba rekordów: 10000
root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)

+------+-----------+--------+-------------------+-------+-------+
|amount|category   |store   |timestamp          |tx_id  |user_id|
+------+-----------+--------+-------------------+-------+-------+
|312.32|elektronika|Warszawa|2026-04-12 08:25:07|TX00001|u48    |
|79.57 |książki    |Warszawa|2026-04-12 08:05:43|TX00002|u15    |
|126.17|odzież     |Warszawa|2026-04-12 09:15:30|TX00003|u18    |
|34.08 |odzież     |Warszawa|2026-04-12 10:05:39|TX00004|u10    |
|428.88|żywność    |Kraków  |2026-04-12 09:04:36|TX00005|u17    |
|345.21|książki    |Warszawa|2026-04-12 09:36:31|TX00006|u25    |
|376.42|żywność    |Warszawa|2026-04-12 10:06:49|TX00007|u15    |
|85.36 |elektronika|Gdańsk  |2026-04-12 09:08:25|TX00008|u24    |
|66.26 |żywno

In [3]:
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn(
    "timestamp",
    to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss")
)

df.printSchema()

root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [4]:
from pyspark.sql.functions import count, sum as _sum, avg, round as _round

store_summary = (
    df.groupBy("store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(avg("amount"), 2).alias("srednia_PLN")
    )
    .orderBy("store")
)

store_summary.show()

+--------+---------+----------+-----------+
|   store|liczba_tx|  suma_PLN|srednia_PLN|
+--------+---------+----------+-----------+
|  Gdańsk|     2498|1021266.35|     408.83|
|  Kraków|     2522|1025896.95|     406.78|
|Warszawa|     2424| 961642.24|     396.72|
| Wrocław|     2556|1002739.21|     392.31|
+--------+---------+----------+-----------+



In [5]:
from pyspark.sql.functions import min as _min, max as _max

category_stats = (
    df.groupBy("category")
    .agg(
        _round(_sum("amount"), 2).alias("suma"),
        _min("amount").alias("min"),
        _max("amount").alias("max"),
    )
    .orderBy("category")
)

category_stats.show()

+-----------+----------+---+-------+
|   category|      suma|min|    max|
+-----------+----------+---+-------+
|elektronika|1520770.69|9.0| 9999.0|
|    książki| 851382.08|5.0|9107.25|
|     odzież| 849877.55|5.0|9696.63|
|    żywność| 789514.43|5.0|6916.92|
+-----------+----------+---+-------+



In [6]:
from pyspark.sql.functions import count, avg

store_stats = (
    df.groupBy("store")
    .agg(
        count("*").alias("liczba_transakcji"),
        avg("amount").alias("srednia_kwota")
    )
    .orderBy("store")
)

store_stats.show()

+--------+-----------------+------------------+
|   store|liczba_transakcji|     srednia_kwota|
+--------+-----------------+------------------+
|  Gdańsk|             2498|  408.833606885508|
|  Kraków|             2522|406.77912371134016|
|Warszawa|             2424| 396.7170957095704|
| Wrocław|             2556|   392.30798513302|
+--------+-----------------+------------------+



In [7]:
top_transactions = df.orderBy(df.amount.desc())

top_transactions.show(10, truncate=False)

+-------+-----------+--------+-------------------+-------+-------+
|amount |category   |store   |timestamp          |tx_id  |user_id|
+-------+-----------+--------+-------------------+-------+-------+
|9999.0 |elektronika|Warszawa|2026-04-12 09:27:06|TX01443|u35    |
|9999.0 |elektronika|Warszawa|2026-04-12 09:37:35|TX09067|u31    |
|9999.0 |elektronika|Gdańsk  |2026-04-12 09:27:54|TX09365|u12    |
|9696.63|odzież     |Wrocław |2026-04-12 08:49:27|TX05828|u10    |
|9107.25|książki    |Gdańsk  |2026-04-12 08:39:02|TX08772|u30    |
|9090.18|książki    |Warszawa|2026-04-12 09:31:39|TX03528|u27    |
|8940.29|elektronika|Wrocław |2026-04-12 09:46:15|TX05891|u15    |
|8199.77|elektronika|Warszawa|2026-04-12 09:48:27|TX04197|u19    |
|7987.53|odzież     |Gdańsk  |2026-04-12 08:06:16|TX08568|u30    |
|7481.32|książki    |Kraków  |2026-04-12 08:42:59|TX09743|u04    |
+-------+-----------+--------+-------------------+-------+-------+
only showing top 10 rows



In [8]:
expensive = df.filter(df.amount > 500)

expensive.show(10)
print("Liczba drogich transakcji:", expensive.count())

+-------+-----------+--------+-------------------+-------+-------+
| amount|   category|   store|          timestamp|  tx_id|user_id|
+-------+-----------+--------+-------------------+-------+-------+
| 660.41|     odzież|  Kraków|2026-04-12 08:29:24|TX00010|    u41|
| 517.07|    żywność|  Kraków|2026-04-12 08:41:13|TX00011|    u21|
| 591.09|    książki| Wrocław|2026-04-12 08:25:43|TX00013|    u48|
| 616.95|    książki|Warszawa|2026-04-12 09:35:53|TX00021|    u43|
|1016.22|     odzież|  Kraków|2026-04-12 08:10:35|TX00025|    u03|
|1696.18|elektronika|Warszawa|2026-04-12 10:24:06|TX00029|    u13|
| 701.06|elektronika|Warszawa|2026-04-12 08:29:56|TX00032|    u14|
|1069.52|    książki|  Kraków|2026-04-12 08:59:47|TX00051|    u43|
| 521.38|    żywność|  Kraków|2026-04-12 09:02:13|TX00056|    u28|
| 577.98|    książki|  Gdańsk|2026-04-12 09:19:31|TX00062|    u33|
+-------+-----------+--------+-------------------+-------+-------+
only showing top 10 rows

Liczba drogich transakcji: 2206


In [9]:
from pyspark.sql.functions import to_timestamp

df = df.withColumn("timestamp", to_timestamp("timestamp"))

df.printSchema()

root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [10]:
from pyspark.sql.functions import hour

hourly = (
    df.withColumn("hour", hour("timestamp"))
    .groupBy("hour")
    .count()
    .orderBy("hour")
)

hourly.show()

+----+-----+
|hour|count|
+----+-----+
|   8| 3150|
|   9| 4661|
|  10| 2189|
+----+-----+



In [12]:
from pyspark.sql.functions import window, count, sum as _sum, round as _round, col


In [13]:
hourly = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window")
)

hourly.show(truncate=False)


+------------------------------------------+---------+----------+
|window                                    |liczba_tx|suma_PLN  |
+------------------------------------------+---------+----------+
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|3150     |1241911.3 |
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|4661     |1896230.21|
|{2026-04-12 10:00:00, 2026-04-12 11:00:00}|2189     |873403.24 |
+------------------------------------------+---------+----------+



In [14]:
(
    hourly
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .show(truncate=False)
)


+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
+-------------------+-------------------+---------+----------+



In [15]:
store_30min = (
    df.groupBy(
        window("timestamp", "30 minutes"),
        "store"
    )
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "store",
        "liczba_tx",
        "suma_PLN",
    )
    .orderBy("od", "store")
)

store_30min.show(truncate=False)


+-------------------+-------------------+--------+---------+---------+
|od                 |do                 |store   |liczba_tx|suma_PLN |
+-------------------+-------------------+--------+---------+---------+
|2026-04-12 08:00:00|2026-04-12 08:30:00|Gdańsk  |252      |93391.22 |
|2026-04-12 08:00:00|2026-04-12 08:30:00|Kraków  |289      |117786.42|
|2026-04-12 08:00:00|2026-04-12 08:30:00|Warszawa|275      |88441.58 |
|2026-04-12 08:00:00|2026-04-12 08:30:00|Wrocław |296      |111540.59|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Gdańsk  |514      |209187.85|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Kraków  |532      |223541.41|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Warszawa|490      |182435.06|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Wrocław |502      |215587.17|
|2026-04-12 09:00:00|2026-04-12 09:30:00|Gdańsk  |619      |253364.95|
|2026-04-12 09:00:00|2026-04-12 09:30:00|Kraków  |590      |224358.03|
|2026-04-12 09:00:00|2026-04-12 09:30:00|Warszawa|584      |214573.66|
|2026-

In [16]:
from pyspark.sql.functions import desc

krakow_hourly = (
    df.filter(col("store") == "Kraków")
    .groupBy(window("timestamp", "1 hour"))
    .agg(
        _round(_sum("amount"), 2).alias("suma_PLN")
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "suma_PLN",
    )
    .orderBy(desc("suma_PLN"))
)

krakow_hourly.show(truncate=False)


+-------------------+-------------------+---------+
|od                 |do                 |suma_PLN |
+-------------------+-------------------+---------+
|2026-04-12 09:00:00|2026-04-12 10:00:00|483309.86|
|2026-04-12 08:00:00|2026-04-12 09:00:00|341327.83|
|2026-04-12 10:00:00|2026-04-12 11:00:00|201259.26|
+-------------------+-------------------+---------+



In [17]:
sliding = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .orderBy("od")
)

sliding.show(truncate=False)


+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 07:30:00|2026-04-12 08:30:00|1112     |411159.81 |
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 08:30:00|2026-04-12 09:30:00|4443     |1753033.6 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 09:30:00|2026-04-12 10:30:00|3696     |1557641.39|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
|2026-04-12 10:30:00|2026-04-12 11:30:00|749      |289709.95 |
+-------------------+-------------------+---------+----------+



In [18]:
tumbling_rows = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(count("tx_id"))
    .count()
)

sliding_rows = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(count("tx_id"))
    .count()
)

print(f"Tumbling (1h):          {tumbling_rows} okien")
print(f"Sliding  (1h / 30min):  {sliding_rows} okien")


Tumbling (1h):          3 okien
Sliding  (1h / 30min):  7 okien


In [ ]:
# Odpowiedz:
# Sliding ma więcej wierszy, ponieważ okna nachodzą na siebie.
# Jedna transakcja może należeć do kilku okien jednocześnie,
# dlatego powstaje więcej grup wynikowych niż w tumbling.


In [ ]:
# 1. Ile transakcji jest w oknie 09:00–10:00?
# ODPOWIEDŹ:
# Sprawdź wartość liczba_tx dla okna 09:00–10:00 w wyniku zadania 3.1.

# 2. Jaka jest różnica między groupBy("store")
#    a groupBy(window(...), "store")?
# ODPOWIEDŹ:
# groupBy("store") grupuje wszystkie transakcje tylko według sklepu,
# bez uwzględniania czasu.
# groupBy(window(...), "store") grupuje transakcje według sklepu
# oraz przedziałów czasowych.

# 3. W oknie sliding 1h/30min — ile okien zawiera transakcje z godziny 09:30?
# ODPOWIEDŹ:
# 2 okna.
# Przykładowo:
# 09:00–10:00
# 09:30–10:30
